# Pearls AQI Predictor - Complete End-to-End Notebook

This single notebook contains the ENTIRE project pipeline so you can
run everything in Google Colab with no local setup required:

1. Install packages
2. Configuration
3. Feature Pipeline (fetch weather + pollution data, engineer features)
4. Historical Backfill (build a training dataset)
5. Exploratory Data Analysis (EDA)
6. Training Pipeline (Ridge Regression, Random Forest, Neural Network)
7. Explainability (SHAP)
8. Save Model to Registry (Hopsworks or local)
9. Inference: 3-Day AQI Forecast + Hazard Alert

**How to use:** Runtime -> Run all. That's it.

Data source: **Open-Meteo** (free, no API key required) - it already
computes the US AQI value for us directly from raw pollutant data.


## 1. Install & Import Packages

In [ ]:
# Install packages that are not pre-installed on Google Colab
!pip install -q hopsworks shap


In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("All libraries imported successfully.")


## 2. Configuration

Change the city coordinates below if you are not predicting AQI for
Sukkur, Pakistan. You can find any city's latitude/longitude on
Google Maps (right-click a location -> the numbers shown are lat, lon).

**Hopsworks (optional but recommended):** Sign up for free at
https://app.hopsworks.ai, create a project, then go to
Account Settings -> API Keys -> create a new key. Paste it below.
If you leave it blank, the notebook automatically works with local
pandas DataFrames instead - everything still works, you just won't
have a real cloud Feature Store / Model Registry.


In [ ]:
# ---- Location settings ----
CITY_NAME = "Sukkur"
LATITUDE = 27.7052
LONGITUDE = 68.8574

# ---- Open-Meteo API endpoints (free, no key needed) ----
AIR_QUALITY_API_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
WEATHER_API_URL = "https://api.open-meteo.com/v1/forecast"

# ---- Hopsworks settings (leave blank to use local fallback) ----
HOPSWORKS_API_KEY = ""   # <-- paste your key here, inside the quotes
HOPSWORKS_PROJECT_NAME = "pearls_aqi"
USE_HOPSWORKS = bool(HOPSWORKS_API_KEY)

# ---- Backfill / forecast settings ----
BACKFILL_PAST_DAYS = 92     # Open-Meteo's free history limit
FORECAST_DAYS = 3
HAZARDOUS_AQI_THRESHOLD = 150  # US AQI: 150+ = Unhealthy

TARGET_COLUMN = "aqi"
FEATURE_COLUMNS = [
    "temperature", "humidity", "wind_speed", "pressure",
    "hour", "day", "month", "day_of_week",
    "aqi_lag_1", "aqi_lag_2", "aqi_change_rate",
]

print(f"Configured for {CITY_NAME} ({LATITUDE}, {LONGITUDE})")
print(f"Using Hopsworks: {USE_HOPSWORKS}")


## 3. Feature Pipeline Functions

These functions fetch raw data and turn it into model-ready features.

**Important - avoiding data leakage:** `aqi_change_rate` is built ONLY
from past values (`aqi_lag_1 - aqi_lag_2`), never from the current
row's own AQI. If we used the current AQI here, the model would
"cheat" during training (looking artificially perfect) and then fail
completely on real forecasting, since we never know the current AQI
in advance in real life.


In [ ]:
def fetch_air_quality(latitude, longitude, past_days=None, forecast_days=None):
    """Fetch hourly air quality data (including ready-made US AQI) from Open-Meteo."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "us_aqi,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,ozone",
        "timezone": "auto",
    }
    if past_days is not None:
        params["past_days"] = past_days
    if forecast_days is not None:
        params["forecast_days"] = forecast_days
    response = requests.get(AIR_QUALITY_API_URL, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


def fetch_weather(latitude, longitude, past_days=None, forecast_days=None):
    """Fetch hourly weather data from Open-Meteo."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure",
        "timezone": "auto",
    }
    if past_days is not None:
        params["past_days"] = past_days
    if forecast_days is not None:
        params["forecast_days"] = forecast_days
    response = requests.get(WEATHER_API_URL, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


def air_quality_json_to_df(aq_json):
    h = aq_json["hourly"]
    return pd.DataFrame({
        "timestamp": pd.to_datetime(h["time"]),
        "aqi": h["us_aqi"],
        "pm10": h["pm10"],
        "pm2_5": h["pm2_5"],
        "carbon_monoxide": h["carbon_monoxide"],
        "nitrogen_dioxide": h["nitrogen_dioxide"],
        "ozone": h["ozone"],
    })


def weather_json_to_df(weather_json):
    h = weather_json["hourly"]
    return pd.DataFrame({
        "timestamp": pd.to_datetime(h["time"]),
        "temperature": h["temperature_2m"],
        "humidity": h["relative_humidity_2m"],
        "wind_speed": h["wind_speed_10m"],
        "pressure": h["surface_pressure"],
    })


def add_time_features(df):
    df = df.copy()
    df["hour"] = df["timestamp"].dt.hour
    df["day"] = df["timestamp"].dt.day
    df["month"] = df["timestamp"].dt.month
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    return df


def add_derived_features(df):
    # Leakage-safe: only uses PAST aqi values, never the current row's own aqi
    df = df.copy()
    df["aqi_lag_1"] = df["aqi"].shift(1)
    df["aqi_lag_2"] = df["aqi"].shift(2)
    df["aqi_change_rate"] = df["aqi_lag_1"] - df["aqi_lag_2"]
    return df


def build_feature_dataframe(aq_json, weather_json):
    aq_df = air_quality_json_to_df(aq_json)
    weather_df = weather_json_to_df(weather_json)
    merged = pd.merge(aq_df, weather_df, on="timestamp", how="inner")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged = add_time_features(merged)
    merged = add_derived_features(merged)
    return merged


def categorize_aqi(aqi_value):
    if aqi_value <= 50: return "Good"
    elif aqi_value <= 100: return "Moderate"
    elif aqi_value <= 150: return "Unhealthy for Sensitive Groups"
    elif aqi_value <= 200: return "Unhealthy"
    elif aqi_value <= 300: return "Very Unhealthy"
    else: return "Hazardous"

print("Feature engineering functions ready.")


## 4. Historical Backfill

This fetches ~92 days of real historical weather + air quality data
for your city and builds the training dataset.


In [ ]:
print(f"Fetching {BACKFILL_PAST_DAYS} days of historical data for {CITY_NAME}...")

aq_json = fetch_air_quality(LATITUDE, LONGITUDE, past_days=BACKFILL_PAST_DAYS)
weather_json = fetch_weather(LATITUDE, LONGITUDE, past_days=BACKFILL_PAST_DAYS)

historical_df = build_feature_dataframe(aq_json, weather_json)
historical_df = historical_df.dropna(subset=["aqi_lag_1", "aqi_lag_2"]).reset_index(drop=True)

print(f"Built {len(historical_df)} rows of historical training data.")
historical_df.head()


In [ ]:
# ---- Save to Hopsworks Feature Store (if configured) ----
if USE_HOPSWORKS:
    import hopsworks
    project = hopsworks.login(api_key_value=HOPSWORKS_API_KEY, project=HOPSWORKS_PROJECT_NAME)
    fs = project.get_feature_store()
    feature_group = fs.get_or_create_feature_group(
        name="aqi_features",
        version=1,
        description="Hourly AQI + weather features for AQI forecasting",
        primary_key=["timestamp"],
        event_time="timestamp",
    )
    feature_group.insert(historical_df, write_options={"wait_for_job": True})
    print("Saved historical data to the Hopsworks Feature Store.")
else:
    print("Hopsworks not configured - historical_df will be used directly "
          "from memory for the rest of this notebook (local fallback mode).")


## 5. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# AQI trend over time
axes[0, 0].plot(historical_df["timestamp"], historical_df["aqi"], linewidth=0.8)
axes[0, 0].set_title(f"AQI Trend Over Time - {CITY_NAME}")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("US AQI")

# AQI distribution
sns.histplot(historical_df["aqi"], bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Distribution of AQI Values")

# Average AQI by hour of day
hourly_avg = historical_df.groupby("hour")["aqi"].mean()
axes[1, 0].bar(hourly_avg.index, hourly_avg.values, color="steelblue")
axes[1, 0].set_title("Average AQI by Hour of Day")
axes[1, 0].set_xlabel("Hour")
axes[1, 0].set_ylabel("Average AQI")

# Correlation heatmap
cols = ["aqi", "temperature", "humidity", "wind_speed", "pressure"]
sns.heatmap(historical_df[cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", ax=axes[1, 1])
axes[1, 1].set_title("Correlation: Weather Variables vs AQI")

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=150)
plt.show()

print("EDA plots saved as 'eda_plots.png' - include this image in your report.")


## 6. Training Pipeline

We train and compare THREE different models, as required by the
project brief:
- **Ridge Regression** (statistical baseline)
- **Random Forest** (tree-based ensemble)
- **Neural Network** (deep learning, via TensorFlow/Keras)

The data is split by **time** (not randomly shuffled) - the oldest
80% is used for training and the newest 20% for testing. This avoids
letting the model "see the future" during evaluation.


In [ ]:
df = historical_df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN]).sort_values("timestamp").reset_index(drop=True)

split_index = int(len(df) * 0.8)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

X_train = train_df[FEATURE_COLUMNS].values
y_train = train_df[TARGET_COLUMN].values
X_test = test_df[FEATURE_COLUMNS].values
y_test = test_df[TARGET_COLUMN].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}")


In [ ]:
def evaluate(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return {"rmse": rmse, "mae": mae, "r2": r2}

results = {}

# ---- Model 1: Ridge Regression ----
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
results["ridge_regression"] = {"model": ridge, "metrics": evaluate(y_test, ridge.predict(X_test_scaled))}

# ---- Model 2: Random Forest ----
rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42)
rf.fit(X_train_scaled, y_train)
results["random_forest"] = {"model": rf, "metrics": evaluate(y_test, rf.predict(X_test_scaled))}

# ---- Model 3: Neural Network ----
from tensorflow import keras
from tensorflow.keras import layers

nn = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),
])
nn.compile(optimizer="adam", loss="mse", metrics=["mae"])
nn.fit(X_train_scaled, y_train, validation_split=0.1, epochs=30, batch_size=32, verbose=0)
nn_preds = nn.predict(X_test_scaled, verbose=0).flatten()
results["neural_network"] = {"model": nn, "metrics": evaluate(y_test, nn_preds)}

print("Model comparison (lower RMSE/MAE is better, higher R^2 is better):")
for name, result in results.items():
    m = result["metrics"]
    print(f"  {name:18s} -> RMSE: {m['rmse']:.3f} | MAE: {m['mae']:.3f} | R^2: {m['r2']:.3f}")

best_name = min(results, key=lambda name: results[name]["metrics"]["rmse"])
best_result = results[best_name]
print(f"\nBest model: {best_name} (RMSE = {best_result['metrics']['rmse']:.3f})")


## 7. Explainability (SHAP)

We use SHAP to explain which features matter most for AQI prediction,
using a Random Forest explainer (tree models work best with SHAP's
fast TreeExplainer).


In [ ]:
import shap

explain_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
explain_model.fit(train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN])

explainer = shap.TreeExplainer(explain_model)
shap_values = explainer.shap_values(train_df[FEATURE_COLUMNS])

shap.summary_plot(shap_values, train_df[FEATURE_COLUMNS])
plt.savefig("shap_summary.png", dpi=150, bbox_inches="tight")
print("SHAP plot saved as 'shap_summary.png' - include this in your report too.")


## 8. Save Best Model to the Model Registry

In [ ]:
import joblib, json, os

os.makedirs("saved_model", exist_ok=True)
joblib.dump(best_result["model"], "saved_model/best_model.pkl")
joblib.dump(scaler, "saved_model/scaler.pkl")
with open("saved_model/model_meta.json", "w") as f:
    json.dump({
        "model_type": best_name,
        "feature_columns": FEATURE_COLUMNS,
        "target_column": TARGET_COLUMN,
        "metrics": best_result["metrics"],
    }, f, indent=2)

if USE_HOPSWORKS:
    mr = project.get_model_registry()
    hops_model = mr.python.create_model(
        name="aqi_predictor_model",
        metrics=best_result["metrics"],
        description=f"Best AQI model ({best_name})",
    )
    hops_model.save("saved_model")
    print("Uploaded model to the Hopsworks Model Registry.")
else:
    print("Saved model locally to the 'saved_model/' folder (local fallback mode).")


## 9. Inference: 3-Day AQI Forecast + Hazard Alert

This is the "web app" logic in notebook form: we fetch the real
upcoming weather forecast, then predict AQI one hour at a time,
feeding each prediction back in as the lag feature for the next hour
(recursive multi-step forecasting).


In [ ]:
forecast_weather_json = fetch_weather(LATITUDE, LONGITUDE, forecast_days=FORECAST_DAYS)
h = forecast_weather_json["hourly"]
forecast_weather_df = pd.DataFrame({
    "timestamp": pd.to_datetime(h["time"]),
    "temperature": h["temperature_2m"],
    "humidity": h["relative_humidity_2m"],
    "wind_speed": h["wind_speed_10m"],
    "pressure": h["surface_pressure"],
})
now = pd.Timestamp.now()
forecast_weather_df = forecast_weather_df[forecast_weather_df["timestamp"] >= now].reset_index(drop=True)

best_model = best_result["model"]
last_known_aqi_1 = historical_df["aqi"].iloc[-1]
last_known_aqi_2 = historical_df["aqi"].iloc[-2]

predictions = []
for _, row in forecast_weather_df.iterrows():
    change_rate = last_known_aqi_1 - last_known_aqi_2
    feature_row = pd.DataFrame([{
        "temperature": row["temperature"], "humidity": row["humidity"],
        "wind_speed": row["wind_speed"], "pressure": row["pressure"],
        "hour": row["timestamp"].hour, "day": row["timestamp"].day,
        "month": row["timestamp"].month, "day_of_week": row["timestamp"].dayofweek,
        "aqi_lag_1": last_known_aqi_1, "aqi_lag_2": last_known_aqi_2,
        "aqi_change_rate": change_rate,
    }])[FEATURE_COLUMNS]

    X_scaled = scaler.transform(feature_row.values)
    predicted_aqi = max(0, float(np.ravel(best_model.predict(X_scaled))[0]))
    predictions.append({"timestamp": row["timestamp"], "predicted_aqi": predicted_aqi})

    last_known_aqi_2 = last_known_aqi_1
    last_known_aqi_1 = predicted_aqi

forecast_df = pd.DataFrame(predictions)
forecast_df["category"] = forecast_df["predicted_aqi"].apply(categorize_aqi)

plt.figure(figsize=(12, 4))
plt.plot(forecast_df["timestamp"], forecast_df["predicted_aqi"], marker="o", markersize=2)
plt.axhline(HAZARDOUS_AQI_THRESHOLD, color="red", linestyle="--", label="Hazardous threshold")
plt.title(f"{FORECAST_DAYS}-Day AQI Forecast - {CITY_NAME}")
plt.xlabel("Time")
plt.ylabel("Predicted US AQI")
plt.legend()
plt.tight_layout()
plt.savefig("forecast_plot.png", dpi=150)
plt.show()

hazardous_hours = forecast_df[forecast_df["predicted_aqi"] >= HAZARDOUS_AQI_THRESHOLD]
if len(hazardous_hours) > 0:
    print(f"ALERT: Hazardous AQI (>= {HAZARDOUS_AQI_THRESHOLD}) predicted starting "
          f"{hazardous_hours['timestamp'].iloc[0]}")
else:
    print("No hazardous AQI levels predicted in the forecast window.")

forecast_df.head(20)


## 10. Next Step: The Live Web Dashboard

This notebook covers the feature/training/inference pipelines and
your report material (EDA + SHAP images saved above). For the
**interactive web dashboard** requirement, use the separate `app.py`
(Streamlit) file provided alongside this notebook:

```bash
streamlit run app.py
```

Or deploy it for free with a public link at
**https://share.streamlit.io** by connecting your GitHub repo -
no server management needed, which keeps the whole project
100% serverless as required.
